# Day 3, Part 2: Call Amazon Bedrock from Python

Send a prompt to a foundation model on Amazon Bedrock with `boto3` and read the answer.

**Before you start**

- AWS credentials on your machine, for example with `aws configure sso`
- Access to a model in your Region (check the Bedrock console)
- A `.env` file next to this notebook (never commit it):

```text
AWS_REGION=us-east-1
BEDROCK_MODEL_ID=<a model ID or inference profile ID you can use>
```

> **Cost:** every call in this notebook is billed. Keep prompts short and `maxTokens` small.

## 1. Setup: the AWS SDK for Python (Boto3)

In [ ]:
%pip install --quiet boto3 python-dotenv

In [ ]:
import os
import time

import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv

load_dotenv()                                   # reads .env into environment variables

REGION = os.getenv("AWS_REGION", "us-east-1")
MODEL_ID = os.getenv("BEDROCK_MODEL_ID")
if not MODEL_ID:
    raise RuntimeError("Set BEDROCK_MODEL_ID in your .env file")

print("Region:", REGION)
print("Model :", MODEL_ID)

## 2. Connect to Amazon Bedrock

Use the **`bedrock-runtime`** client to run models. (The separate `bedrock` client lists models and manages settings.)
`boto3` finds your credentials on its own, so there is no key in the code.

In [ ]:
client = boto3.client("bedrock-runtime", region_name=REGION)

## 3. Send a prompt

The **Converse** API uses the same request shape for every supported model.

- `messages`: the conversation. Each message has a `role` and a list of content blocks.
- `inferenceConfig`: `maxTokens` caps the answer length (and cost), `temperature` controls randomness.

In [ ]:
response = client.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": "In one sentence, what is Amazon Bedrock?"}]},
    ],
    inferenceConfig={"maxTokens": 200, "temperature": 0.5},
)

## 4. Read the response

In [ ]:
reply = response["output"]["message"]["content"][0]["text"]

print(reply)
print("Tokens    :", response["usage"])
print("Stop reason:", response["stopReason"])

| Key | Meaning |
|---|---|
| `output.message.content` | The answer: a list of blocks, the text is in `["text"]` |
| `usage` | Input and output tokens. This is what you pay for. |
| `stopReason` | `end_turn` means finished. `max_tokens` means the answer was cut off. |

## 5. Handle errors and retries

A call can fail. `boto3` raises a `ClientError`, and the error name is in `error.response["Error"]["Code"]`.

- **Temporary** errors (`ThrottlingException`: too many requests) are worth retrying after a short wait.
- **Permanent** errors (`AccessDeniedException`, `ValidationException`) will not fix themselves. Show the error.

In [ ]:
def ask(prompt, max_attempts=3):
    """Send one prompt and return the reply text, retrying when Bedrock is busy."""
    for attempt in range(1, max_attempts + 1):
        try:
            response = client.converse(
                modelId=MODEL_ID,
                messages=[{"role": "user", "content": [{"text": prompt}]}],
                inferenceConfig={"maxTokens": 200},
            )
            return response["output"]["message"]["content"][0]["text"]
        except ClientError as error:
            code = error.response["Error"]["Code"]
            if code == "ThrottlingException" and attempt < max_attempts:
                wait = 2 ** attempt                       # 2s, then 4s
                print(f"Busy. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                print("Bedrock error:", code)
                raise


print(ask("Give me one tip for writing a good prompt."))

## 6. Manage credentials safely

`boto3` looks for credentials in this order and uses the first it finds:
code arguments, then **environment variables**, then your **profile or SSO login** (`~/.aws/`), then the **IAM role** of the server it runs on.

| Variable | Purpose |
|---|---|
| `AWS_PROFILE` | Which saved profile to use |
| `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN` | A key pair, for temporary credentials |
| `AWS_DEFAULT_REGION` | Default Region |

Three rules:

1. **Never** put keys in code or notebooks. Use SSO, a profile, a role, or `.env`.
2. Add `.env` to `.gitignore`.
3. Give the identity only the permission it needs (`bedrock:InvokeModel` on the model you use).

## 7. Direct API vs. Amazon Bedrock

| | Direct model API | Amazon Bedrock |
|---|---|---|
| Sign-in | Provider API key | AWS identity (IAM, SSO, roles) |
| SDK | The provider's SDK | `boto3` |
| Billing | Provider account | Your AWS bill |
| Control | API key settings | IAM, CloudTrail, private networking |
| Models | One provider | Many providers, one Converse API |

Use **Bedrock** when you already work in AWS and want its security and billing. Use the **direct API** for the quickest start or the newest features.
Feature availability changes, so check the current documentation.

## Try it

1. Change `temperature` to `0` and then `1`, and run the same prompt twice. What differs?
2. Set `maxTokens` to `10`. What is the `stopReason`?
3. Add a `system` prompt (`system=[{"text": "Answer like a pirate."}]`) to the `converse` call.